# GLiNER2 Medical NER — Model Training & Evaluation

This notebook trains a **GLiNER2** model on the prepared medical NER dataset using **LoRA** (Imbalance Strategy #5) for parameter-efficient fine-tuning.

## Prerequisites
- Run `data_preparation.ipynb` first to generate the JSONL files
- Required files in `prepared_data/`: `train.jsonl`, `val.jsonl`, `test.jsonl`, `eval_ood.jsonl`

## Notebook Overview

| Step | Description |
|------|-------------|
| 1. Setup | Install dependencies, check GPU |
| 2. Configuration | All training hyperparameters |
| 3. Load Data | Load & validate prepared JSONL files |
| 4. Model Setup | Load GLiNER2 base model |
| 5. Training | LoRA fine-tuning (Strategy #5) |
| 6. In-Distribution Evaluation | Evaluate on held-out test split (P/R/F1) |
| 7. OOD Evaluation | Out-of-distribution evaluation (P/R/F1) |
| 8. Push to HuggingFace | Upload trained model/adapter to HF Hub |
| 9. Inference | Example usage of the trained model |

In [1]:
# Install dependencies
# Uncomment when running on Google Colab
%pip install -q gliner2 "transformers<4.48" huggingface_hub

import json
import os
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("On Colab: Runtime → Change runtime type → GPU")

Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA GeForce GTX TITAN X
VRAM: 12.8 GB


## 1. Configuration

All training hyperparameters in one place.

### Why LoRA? (Imbalance Strategy #5)

LoRA (Low-Rank Adaptation) is used because:
1. **Limited Colab GPU memory** — LoRA trains ~0.1-1% of parameters
2. **Prevents overfitting** — with limited labeled data (~542 docs), full fine-tuning risks overfitting
3. **Regularization effect** — the low-rank constraint acts as implicit regularization
4. **Fast training** — 2-3x faster than full fine-tuning
5. **Small checkpoint** — ~5-10 MB adapter vs ~450 MB full model

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

# --- Paths ---
DATA_DIR = "./prepared_data"                     # Output from data_preparation.ipynb
TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")
TEST_FILE = os.path.join(DATA_DIR, "test.jsonl")
OOD_EVAL_FILE = os.path.join(DATA_DIR, "eval_ood.jsonl")
OUTPUT_DIR = "./gliner2_medical_ner"              # Model output directory

# --- Base Model ---
BASE_MODEL = "fastino/gliner2-base-v1"           # 205M params

# --- Training Hyperparameters ---
NUM_EPOCHS = 15
BATCH_SIZE = 2                                    # Reduced from 8 to avoid OOM on 12GB VRAM
GRADIENT_ACCUMULATION_STEPS = 16                  # Effective batch = 2 x 16 = 32 (same as before)
TASK_LR = 5e-4
ENCODER_LR = 1e-5                                # Used only for full fine-tuning
WARMUP_RATIO = 0.1
SCHEDULER = "cosine"
WEIGHT_DECAY = 0.01

# --- LoRA (Strategy #5) ---
USE_LORA = True
LORA_R = 16                                       # Rank (higher = more params)
LORA_ALPHA = 32                                   # Scaling (typically 2*r)
LORA_DROPOUT = 0.1                                # Regularization
LORA_TARGET_MODULES = ["encoder", "span_rep", "classifier"]
SAVE_ADAPTER_ONLY = True

# --- Memory Optimization ---
FP16 = True
NUM_WORKERS = 2

# --- Evaluation & Checkpointing ---
EVAL_STRATEGY = "epoch"
SAVE_BEST = True
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 3
LOGGING_STEPS = 20

# --- HuggingFace Hub ---
HF_REPO_ID = "haiderAI/gliner2-medical-dataset-ner"   # Change this!
HF_PRIVATE = True                                            # Private repo

# --- Reproducibility ---
SEED = 42

# --- Entity types (must match data_preparation.ipynb) ---
ENTITY_TYPES = ["Dataset"]
ENTITY_DESCRIPTIONS = {
    "Dataset": "Names of datasets, databases, corpus, collection,  benchmarks, or data collections used in scientific research"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Base model: {BASE_MODEL}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Training: {NUM_EPOCHS} epochs, batch={BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS}={BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS}")
print(f"Output: {OUTPUT_DIR}")

Base model: fastino/gliner2-base-v1
LoRA: r=16, alpha=32, dropout=0.1
Training: 15 epochs, batch=2x16=32
Output: ./gliner2_medical_ner


## 2. Load & Validate Data

Load the JSONL files generated by `data_preparation.ipynb` and validate them.

In [3]:
# Quick check files exist
for fpath in [TRAIN_FILE, VAL_FILE, TEST_FILE, OOD_EVAL_FILE]:
    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f"File not found: {fpath}\n"
            f"Run data_preparation.ipynb first to generate the JSONL files."
        )

# Count examples
def count_jsonl(path):
    count = 0
    pos = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                count += 1
                ex = json.loads(line)
                if any(len(v) > 0 for v in ex.get('output', {}).get('entities', {}).values()):
                    pos += 1
    return count, pos

for name, path in [("Train", TRAIN_FILE), ("Val", VAL_FILE), ("Test (ID)", TEST_FILE), ("OOD Eval", OOD_EVAL_FILE)]:
    total, pos = count_jsonl(path)
    print(f"{name:10s}: {total:5d} examples ({pos} positive, {total-pos} negative)")

print("\n✓ All data files present and readable.")

Train     :  5133 examples (4060 positive, 1073 negative)
Val       :   684 examples (562 positive, 122 negative)
Test (ID) :  1028 examples (823 positive, 205 negative)
OOD Eval  :  1129 examples (246 positive, 883 negative)

✓ All data files present and readable.


In [4]:
# GLiNER2 validation (optional but recommended)
from gliner2.training.data import TrainingDataset

for label, path in [("Training", TRAIN_FILE), ("Validation", VAL_FILE)]:
    print(f"Loading and validating {label.lower()} data...")
    ds = TrainingDataset.load(path)
    try:
        ds.validate()
    except TypeError:
        pass
    if hasattr(ds, 'print_stats'):
        ds.print_stats()
    else:
        print(f"  {label}: {len(ds)} examples loaded ✓")
    print()

print("✓ Datasets validated.")

Loading and validating training data...
Loaded 5133 examples from prepared_data/train.jsonl

GLiNER2 Training Dataset Statistics
Total examples: 5133

Text lengths: min=338, max=1500, mean=1225.7

Task Distribution:
  entities_only: 5133 (100.0%)

Entity Types (7450 total mentions):
  Dataset: 7450


Loading and validating validation data...
Loaded 684 examples from prepared_data/val.jsonl

GLiNER2 Training Dataset Statistics
Total examples: 684

Text lengths: min=325, max=1500, mean=1187.0

Task Distribution:
  entities_only: 684 (100.0%)

Entity Types (1077 total mentions):
  Dataset: 1077


✓ Datasets validated.


## 3. Load GLiNER2 Model

Load the pre-trained base model that we'll fine-tune with LoRA.

In [5]:
from gliner2 import GLiNER2

print(f"Loading base model: {BASE_MODEL}")
model = GLiNER2.from_pretrained(BASE_MODEL)
print("✓ Model loaded.")

# Quick sanity check — run inference before training
test_text = "We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark."
result = model.extract_entities(test_text, ENTITY_TYPES)
print(f"\nPre-training sanity check:")
print(f"  Input: {test_text}")
print(f"  Output: {result}")

Loading base model: fastino/gliner2-base-v1
🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
✓ Model loaded.

Pre-training sanity check:
  Input: We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark.
  Output: {'entities': {'Dataset': ['CIFAR-10', 'ImageNet']}}


## 4. Configure & Run Training

### Strategy #5 — LoRA for Regularization

LoRA adds low-rank trainable matrices to the model's attention layers while keeping the original weights frozen. This:
- Reduces trainable parameters from ~205M to ~1-2M
- Acts as a regularizer against overfitting on sparse entity data
- Enables training on Colab's T4 GPU with comfortable memory headroom

### Key Training Decisions
- **Cosine scheduler** — smooth learning rate decay for better convergence
- **Early stopping (patience=3)** — stops if validation loss doesn't improve for 3 epochs
- **FP16 + gradient checkpointing** — memory optimization for Colab
- **Gradient accumulation** — effective batch size of 32 with actual batch of 8

In [6]:
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

config = TrainingConfig(
    # Output
    output_dir=OUTPUT_DIR,
    experiment_name="medical_dataset_ner",

    # Training
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Learning rates
    encoder_lr=ENCODER_LR,
    task_lr=TASK_LR,

    # Scheduler
    warmup_ratio=WARMUP_RATIO,
    scheduler_type=SCHEDULER,
    weight_decay=WEIGHT_DECAY,

    # LoRA (Strategy #5)
    use_lora=USE_LORA,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODULES,
    save_adapter_only=SAVE_ADAPTER_ONLY,

    # Memory optimization
    fp16=FP16,
    num_workers=NUM_WORKERS,

    # Evaluation & checkpointing
    eval_strategy=EVAL_STRATEGY,
    save_best=SAVE_BEST,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,

    # Logging
    logging_steps=LOGGING_STEPS,

    # Reproducibility
    seed=SEED,
)

print("Training configuration:")
print(f"  Epochs:          {config.num_epochs}")
print(f"  Batch size:      {config.batch_size} x {config.gradient_accumulation_steps} = {config.batch_size * config.gradient_accumulation_steps}")
print(f"  LoRA:            r={config.lora_r}, alpha={config.lora_alpha}, dropout={config.lora_dropout}")
print(f"  LoRA targets:    {config.lora_target_modules}")
print(f"  Task LR:         {config.task_lr}")
print(f"  Scheduler:       {config.scheduler_type} (warmup {config.warmup_ratio})")
print(f"  Early stopping:  patience={config.early_stopping_patience}")
print(f"  FP16:            {config.fp16}")

Training configuration:
  Epochs:          15
  Batch size:      2 x 16 = 32
  LoRA:            r=16, alpha=32, dropout=0.1
  LoRA targets:    ['encoder', 'span_rep', 'classifier']
  Task LR:         0.0005
  Scheduler:       cosine (warmup 0.1)
  Early stopping:  patience=3
  FP16:            True


In [7]:
# Train!
trainer = GLiNER2Trainer(model, config)

print("Starting training...")
print("="*60)

results = trainer.train(
    train_data=TRAIN_FILE,
    eval_data=VAL_FILE,
)

print("="*60)
print("Training complete!")
print(f"  Best validation loss: {results.get('best_metric', 'N/A')}")
print(f"  Total steps:          {results.get('total_steps', 'N/A')}")
total_time = results.get('total_time_seconds', 0)
if total_time:
    print(f"  Training time:        {total_time/60:.1f} minutes")

2026-03-11 07:53:21 - INFO - gliner2.training.trainer - Setting up LoRA for parameter-efficient fine-tuning...
2026-03-11 07:53:21 - INFO - gliner2.training.trainer - Froze all model parameters for LoRA training
2026-03-11 07:53:21 - INFO - gliner2.training.lora - Applied LoRA to 80 layers
2026-03-11 07:53:21 - INFO - gliner2.training.trainer - LoRA setup complete: 3,096,592 trainable params out of 211,573,413 total (1.46%)


🔧 LoRA Configuration
Enabled            : True
Rank (r)           : 16
Alpha              : 32
Scaling (α/r)      : 2.0000
Dropout            : 0.1
Target modules     : encoder, span_rep, classifier
LoRA layers        : 80
----------------------------------------------------------------------
Trainable params   : 3,096,592 / 211,573,413 (1.46%)
Memory savings     : ~98.5% fewer gradients
Starting training...


Validating records: 100%|██████████| 5133/5133 [00:00<00:00, 89491.64record/s]
2026-03-11 07:53:22 - INFO - gliner2.training.trainer - Optimizer: LoRA params only = 160, LR=0.0005
2026-03-11 07:53:22 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Num examples = 5133
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Num epochs = 15
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Batch size = 2
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Gradient accumulation steps = 16
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Effective batch size = 32
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Total optimization steps = 2400
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   Warmup steps = 240
2026-03-11 07:53:22 - INFO - gliner2.training.trainer -   LoRA enabled: 3,096,592 trainable / 211,573,413 total (1.46%)


Training:   0%|          | 0/2400 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the lear

...

Evaluating:   0%|          | 0/86 [00:00<?, ?it/s]

2026-03-11 08:17:46 - INFO - gliner2.training.lora - Saved 160 LoRA tensors to gliner2_medical_ner/best/adapter_weights.safetensors
2026-03-11 08:17:46 - INFO - gliner2.training.lora - Saved adapter config to gliner2_medical_ner/best/adapter_config.json
2026-03-11 08:17:46 - INFO - gliner2.training.lora - Saved LoRA adapter to gliner2_medical_ner/best
2026-03-11 08:17:46 - INFO - gliner2.training.trainer - 💾 Saved adapter checkpoint 'best' | step 322 | epoch 2.0 | 3,096,592 params | 11.8MB | 0.3s
2026-03-11 08:17:46 - INFO - gliner2.training.trainer - New best eval_loss: 16.8583
2026-03-11 08:17:46 - INFO - gliner2.training.lora - Saved 160 LoRA tensors to gliner2_medical_ner/checkpoint-epoch-2/adapter_weights.safetensors
2026-03-11 08:17:46 - INFO - gliner2.training.lora - Saved adapter config to gliner2_medical_ner/checkpoint-epoch-2/adapter_config.json
2026-03-11 08:17:46 - INFO - gliner2.training.lora - Saved LoRA adapter to gliner2_medical_ner/checkpoint-epoch-2
2026-03-11 08:17:4

Evaluating:   0%|          | 0/86 [00:00<?, ?it/s]

2026-03-11 08:30:01 - INFO - gliner2.training.lora - Saved 160 LoRA tensors to gliner2_medical_ner/best/adapter_weights.safetensors
2026-03-11 08:30:01 - INFO - gliner2.training.lora - Saved adapter config to gliner2_medical_ner/best/adapter_config.json
2026-03-11 08:30:01 - INFO - gliner2.training.lora - Saved LoRA adapter to gliner2_medical_ner/best
2026-03-11 08:30:01 - INFO - gliner2.training.trainer - 💾 Saved adapter checkpoint 'best' | step 483 | epoch 3.0 | 3,096,592 params | 11.8MB | 0.3s
2026-03-11 08:30:01 - INFO - gliner2.training.trainer - New best eval_loss: 13.8095
2026-03-11 08:30:01 - INFO - gliner2.training.trainer - Early stopping triggered at epoch 3
2026-03-11 08:30:01 - INFO - gliner2.training.lora - Saved 160 LoRA tensors to gliner2_medical_ner/final/adapter_weights.safetensors
2026-03-11 08:30:01 - INFO - gliner2.training.lora - Saved adapter config to gliner2_medical_ner/final/adapter_config.json
2026-03-11 08:30:01 - INFO - gliner2.training.lora - Saved LoRA ad

Training complete!
  Best validation loss: 13.809467342010764
  Total steps:          483
  Training time:        36.7 minutes


## 5. Load Best Model

Load the best checkpoint (selected by lowest validation loss during training).

In [10]:
# The model object is already fine-tuned in-place after trainer.train()
# GLiNER2.from_pretrained() has a bug with local paths (passes file path to
# transformers' from_pretrained which rejects paths with >1 slash as repo IDs).
# Workaround: use the already-trained model directly.

best_model = model
print("✓ Using trained model (modified in-place by LoRA training).")

# Verify checkpoint was saved (for later reuse / HF upload)
best_path = os.path.abspath(os.path.join(OUTPUT_DIR, "best"))
final_path = os.path.abspath(os.path.join(OUTPUT_DIR, "final"))
if os.path.exists(best_path):
    print(f"  Best checkpoint saved at: {best_path}")
    print(f"  Files: {os.listdir(best_path)}")
elif os.path.exists(final_path):
    print(f"  Final checkpoint saved at: {final_path}")
    print(f"  Files: {os.listdir(final_path)}")
else:
    print("  WARNING: No checkpoint directory found. Model exists only in memory.")

# Quick sanity check
test_text = "We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark."
result = best_model.extract_entities(test_text, ENTITY_TYPES)
print(f"\nPost-training sanity check:")
print(f"  Input: {test_text}")
print(f"  Output: {result}")

✓ Using trained model (modified in-place by LoRA training).
  Best checkpoint saved at: /work/RnD/gliner2_medical_ner/best
  Files: ['adapter_weights.safetensors', 'adapter_config.json']

Post-training sanity check:
  Input: We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark.
  Output: {'entities': {'Dataset': ['CIFAR-10 dataset', 'ImageNet benchmark']}}


## 6. In-Distribution & Out-of-Distribution Evaluation

We evaluate the trained model on **two separate test sets**:

1. **In-Distribution (ID) Test Set** — 15% held-out split from the same data source. This measures how well the model generalizes to unseen chunks from the same distribution. *Not used during training or checkpoint selection.*

2. **Out-of-Distribution (OOD) Test Set** — 44 separate documents. This measures how well the model generalizes to entirely new documents.

### Metrics
- **Exact Match** — predicted entity text must exactly match the ground truth
- **Partial Match** — predicted entity text overlaps with ground truth (substring)
- **Precision** — fraction of predictions that are correct
- **Recall** — fraction of ground truth entities that were found
- **F1** — harmonic mean of precision and recall

In [11]:
def load_eval_data(filepath):
    """Load JSONL evaluation data."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def evaluate_model(model, eval_data, entity_types):
    """
    Evaluate the model on evaluation data.
    Returns per-type and overall metrics.
    """
    all_true_entities = []   # list of sets of (entity_type, entity_text)
    all_pred_entities = []   # list of sets of (entity_type, entity_text)

    for example in eval_data:
        text = example["input"]
        gold_entities = example["output"]["entities"]

        # Ground truth
        true_set = set()
        for etype, mentions in gold_entities.items():
            for mention in mentions:
                true_set.add((etype, mention.strip().lower()))

        # Predictions
        pred_result = model.extract_entities(text, entity_types)
        pred_set = set()
        pred_entities = pred_result.get("entities", {})
        for etype, mentions in pred_entities.items():
            if isinstance(mentions, list):
                for mention in mentions:
                    if isinstance(mention, dict):
                        mention = mention.get("text", "")
                    pred_set.add((etype, str(mention).strip().lower()))

        all_true_entities.append(true_set)
        all_pred_entities.append(pred_set)

    # Compute metrics
    tp = fp = fn = 0
    partial_tp = 0

    for true_set, pred_set in zip(all_true_entities, all_pred_entities):
        # Exact match
        tp += len(true_set & pred_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)

        # Partial match: check if any predicted mention is a substring of a true mention or vice versa
        for ptype, pmention in pred_set:
            if (ptype, pmention) not in true_set:
                for ttype, tmention in true_set:
                    if ptype == ttype and (pmention in tmention or tmention in pmention):
                        partial_tp += 1
                        break

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    partial_precision = (tp + partial_tp) / max(tp + partial_tp + fp - partial_tp, 1)
    partial_recall = (tp + partial_tp) / max(tp + partial_tp + fn - partial_tp, 1)
    partial_f1 = 2 * partial_precision * partial_recall / max(partial_precision + partial_recall, 1e-8)

    metrics = {
        "exact_match": {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn},
        "partial_match": {"precision": partial_precision, "recall": partial_recall, "f1": partial_f1, "partial_tp": partial_tp},
        "total_true": tp + fn,
        "total_pred": tp + fp,
    }
    return metrics

print("Evaluation functions defined.")

Evaluation functions defined.


In [12]:
def print_metrics(metrics, label):
    """Pretty-print evaluation metrics."""
    print("=" * 60)
    print(f"{label} EVALUATION RESULTS")
    print("=" * 60)
    print(f"\nTotal ground truth entities: {metrics['total_true']}")
    print(f"Total predicted entities:    {metrics['total_pred']}")

    print(f"\n--- Exact Match ---")
    em = metrics['exact_match']
    print(f"  Precision: {em['precision']:.4f}")
    print(f"  Recall:    {em['recall']:.4f}")
    print(f"  F1:        {em['f1']:.4f}")
    print(f"  (TP={em['tp']}, FP={em['fp']}, FN={em['fn']})")

    print(f"\n--- Partial Match ---")
    pm = metrics['partial_match']
    print(f"  Precision: {pm['precision']:.4f}")
    print(f"  Recall:    {pm['recall']:.4f}")
    print(f"  F1:        {pm['f1']:.4f}")


# --- In-Distribution Test Set ---
print("Evaluating on in-distribution test set...")
id_data = load_eval_data(TEST_FILE)
id_metrics = evaluate_model(best_model, id_data, ENTITY_TYPES)
print_metrics(id_metrics, "IN-DISTRIBUTION (ID)")

print("\n")

# --- Out-of-Distribution Test Set ---
print("Evaluating on out-of-distribution test set...")
ood_data = load_eval_data(OOD_EVAL_FILE)
ood_metrics = evaluate_model(best_model, ood_data, ENTITY_TYPES)
print_metrics(ood_metrics, "OUT-OF-DISTRIBUTION (OOD)")

# --- Side-by-side comparison ---
print("\n" + "=" * 60)
print("COMPARISON: ID vs OOD")
print("=" * 60)
print(f"{'Metric':<20} {'ID':>10} {'OOD':>10}")
print("-" * 40)
print(f"{'Exact P':<20} {id_metrics['exact_match']['precision']:>10.4f} {ood_metrics['exact_match']['precision']:>10.4f}")
print(f"{'Exact R':<20} {id_metrics['exact_match']['recall']:>10.4f} {ood_metrics['exact_match']['recall']:>10.4f}")
print(f"{'Exact F1':<20} {id_metrics['exact_match']['f1']:>10.4f} {ood_metrics['exact_match']['f1']:>10.4f}")
print(f"{'Partial P':<20} {id_metrics['partial_match']['precision']:>10.4f} {ood_metrics['partial_match']['precision']:>10.4f}")
print(f"{'Partial R':<20} {id_metrics['partial_match']['recall']:>10.4f} {ood_metrics['partial_match']['recall']:>10.4f}")
print(f"{'Partial F1':<20} {id_metrics['partial_match']['f1']:>10.4f} {ood_metrics['partial_match']['f1']:>10.4f}")

# Save all metrics
all_metrics = {"in_distribution": id_metrics, "out_of_distribution": ood_metrics}
metrics_path = os.path.join(OUTPUT_DIR, "eval_metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"\nAll metrics saved to {metrics_path}")

Evaluating on in-distribution test set...
IN-DISTRIBUTION (ID) EVALUATION RESULTS

Total ground truth entities: 1496
Total predicted entities:    1515

--- Exact Match ---
  Precision: 0.7954
  Recall:    0.8055
  F1:        0.8004
  (TP=1205, FP=310, FN=291)

--- Partial Match ---
  Precision: 0.8561
  Recall:    0.8670
  F1:        0.8615


Evaluating on out-of-distribution test set...
OUT-OF-DISTRIBUTION (OOD) EVALUATION RESULTS

Total ground truth entities: 436
Total predicted entities:    537

--- Exact Match ---
  Precision: 0.5363
  Recall:    0.6606
  F1:        0.5920
  (TP=288, FP=249, FN=148)

--- Partial Match ---
  Precision: 0.6331
  Recall:    0.7798
  F1:        0.6989

COMPARISON: ID vs OOD
Metric                       ID        OOD
----------------------------------------
Exact P                  0.7954     0.5363
Exact R                  0.8055     0.6606
Exact F1                 0.8004     0.5920
Partial P                0.8561     0.6331
Partial R                0.

In [13]:
# Qualitative examples — show predictions vs ground truth
def show_examples(data, label, model, entity_types, n=3):
    print(f"\n{'=' * 60}")
    print(f"QUALITATIVE EXAMPLES — {label} (first {n} with entities)")
    print(f"{'=' * 60}")
    shown = 0
    for ex in data:
        gold = ex["output"]["entities"]
        has_gold = any(len(v) > 0 for v in gold.values())
        if not has_gold:
            continue
        text = ex["input"]
        pred = model.extract_entities(text, entity_types)
        print(f"\n--- Example {shown+1} ---")
        print(f"Text (first 300 chars): {text[:300]}...")
        print(f"Ground truth: {gold}")
        print(f"Predicted:    {pred.get('entities', {})}")
        shown += 1
        if shown >= n:
            break

show_examples(id_data, "IN-DISTRIBUTION", best_model, ENTITY_TYPES, n=3)
show_examples(ood_data, "OUT-OF-DISTRIBUTION", best_model, ENTITY_TYPES, n=3)


QUALITATIVE EXAMPLES — IN-DISTRIBUTION (first 3 with entities)

--- Example 1 ---
Text (first 300 chars): Section 2.2. This is followed by Sections 2.3 and 2.4, which detail our two training stages that use conditional probability and a numerically stable unconditional probability formulation, respectively. Datasets and Taxonomy The first step in creating an HMLC system is to create the label taxonomy. ...
Ground truth: {'Dataset': ['PLCO dataset']}
Predicted:    {'Dataset': ['PLCO dataset', 'NegBio']}

--- Example 2 ---
Text (first 300 chars): n reproducibility in multiple datasets (the "overlap analysis") is also applicable to other network reconstruction strategies. It has been stated by leaders in artificial intelligence and data mining that "invariably, simple models and a lot of data trump more elaborate models based on less data" . ...
Ground truth: {'Dataset': ['HIPPIE']}
Predicted:    {'Dataset': ['HIPPIE']}

--- Example 3 ---
Text (first 300 chars): capes experiments in this

## 7. Push to HuggingFace Hub

Upload the trained model (or LoRA adapter) to a HuggingFace repository for easy sharing and deployment.

### Authentication
You need a HuggingFace token with **write access**:
1. Go to https://huggingface.co/settings/tokens
2. Create a token with "Write" permissions
3. Set it below or use `huggingface-cli login`

In [16]:
from huggingface_hub import HfApi, login

# Option 1: Login interactively (recommended on Colab)
# This will prompt you for your HF token
login()

# Option 2: Login with token directly (uncomment and set your token)
# login(token="hf_YOUR_TOKEN_HERE")

print("✓ Authenticated with HuggingFace Hub.")

✓ Authenticated with HuggingFace Hub.


In [17]:
# Upload model to HuggingFace Hub
api = HfApi()

# Determine which directory to upload (best checkpoint)
model_dir = os.path.join(OUTPUT_DIR, "best")
if not os.path.exists(model_dir):
    model_dir = os.path.join(OUTPUT_DIR, "final")

print(f"Uploading model from: {model_dir}")
print(f"Target repo: {HF_REPO_ID}")
print(f"Private: {HF_PRIVATE}")

# Create repo if it doesn't exist
api.create_repo(
    repo_id=HF_REPO_ID,
    private=HF_PRIVATE,
    exist_ok=True,
)

# Upload model files
api.upload_folder(
    folder_path=model_dir,
    repo_id=HF_REPO_ID,
    commit_message="Upload GLiNER2 medical NER model (LoRA fine-tuned)",
)

# Also upload eval metrics
metrics_file = os.path.join(OUTPUT_DIR, "eval_metrics.json")
if os.path.exists(metrics_file):
    api.upload_file(
        path_or_fileobj=metrics_file,
        path_in_repo="eval_metrics.json",
        repo_id=HF_REPO_ID,
        commit_message="Upload evaluation metrics (ID + OOD)",
    )

print(f"\n✓ Model uploaded to: https://huggingface.co/{HF_REPO_ID}")

Uploading model from: ./gliner2_medical_ner/best
Target repo: haiderAI/gliner2-medical-dataset-ner
Private: True


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✓ Model uploaded to: https://huggingface.co/haiderAI/gliner2-medical-dataset-ner


In [18]:
# Create and upload a model card (README.md) with training details

model_card = f"""---
tags:
  - gliner2
  - ner
  - medical
  - lora
  - entity-extraction
license: apache-2.0
---

# GLiNER2 Medical NER — Dataset Entity Extraction

Fine-tuned [GLiNER2]({BASE_MODEL}) model for extracting **dataset names** from medical/scientific research papers.

## Entity Types

| Entity Type | Description |
|-------------|-------------|
| `Dataset` | {ENTITY_DESCRIPTIONS.get('Dataset', 'Dataset names')} |

## Training Details

- **Base model**: `{BASE_MODEL}`
- **Method**: LoRA (r={LORA_R}, alpha={LORA_ALPHA})
- **Epochs**: {NUM_EPOCHS}
- **Batch size**: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS}
- **Learning rate**: {TASK_LR}
- **Scheduler**: {SCHEDULER}

## Evaluation

Evaluated on two separate test sets:
- **In-Distribution (ID)**: 15% held-out split from same data source (not used during training)
- **Out-of-Distribution (OOD)**: 44 separate documents

See `eval_metrics.json` for detailed P/R/F1 metrics.

## Usage

```python
from gliner2 import GLiNER2

model = GLiNER2.from_pretrained("{HF_REPO_ID}")

text = "We evaluated our method on the CIFAR-10 and ImageNet datasets."
result = model.extract_entities(text, ["Dataset"])
print(result)
```
"""

# Save locally and upload
card_path = os.path.join(OUTPUT_DIR, "MODEL_CARD.md")
with open(card_path, 'w', encoding='utf-8') as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj=card_path,
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    commit_message="Add model card",
)

print(f"✓ Model card uploaded to: https://huggingface.co/{HF_REPO_ID}")

✓ Model card uploaded to: https://huggingface.co/haiderAI/gliner2-medical-dataset-ner


## 8. Inference Examples

Demonstrate how to use the trained model for inference on new texts.

In [20]:
# Example usage of the trained model

test_texts = [
    "We used the MIMIC-III database and PhysioNet data for patient outcome prediction.",
    "The model was benchmarked on CIFAR-10, CIFAR-100, and the full ImageNet dataset.",
    "Our ECG classification approach was tested on the MIT-BIH Arrhythmia Database and the PTB Diagnostic ECG Database.",
    "Feature extraction was performed using standard NLP techniques without any specific dataset.",
]

print("=" * 60)
print("INFERENCE EXAMPLES")
print("=" * 60)

for i, text in enumerate(test_texts):
    # Basic extraction
    result = best_model.extract_entities(text, ENTITY_TYPES)

    # With descriptions for better accuracy
    result_desc = best_model.extract_entities(text, ENTITY_DESCRIPTIONS)

    print(f"\n--- Example {i+1} ---")
    print(f"  Text: {text}")
    print(f"  Entities (basic):        {result.get('entities', {})}")
    print(f"  Entities (w/ desc):      {result_desc.get('entities', {})}")

print("\n" + "=" * 60)
print("Done! Model is ready for deployment.")
print(f"Load from HF: GLiNER2.from_pretrained('{HF_REPO_ID}')")

INFERENCE EXAMPLES

--- Example 1 ---
  Text: We used the MIMIC-III database and PhysioNet data for patient outcome prediction.
  Entities (basic):        {'Dataset': ['MIMIC-III database', 'PhysioNet data']}
  Entities (w/ desc):      {'Dataset': ['MIMIC-III database', 'PhysioNet data']}

--- Example 2 ---
  Text: The model was benchmarked on CIFAR-10, CIFAR-100, and the full ImageNet dataset.
  Entities (basic):        {'Dataset': ['CIFAR-100', 'CIFAR-10', 'ImageNet dataset']}
  Entities (w/ desc):      {'Dataset': ['CIFAR-10', 'CIFAR-100', 'ImageNet dataset']}

--- Example 3 ---
  Text: Our ECG classification approach was tested on the MIT-BIH Arrhythmia Database and the PTB Diagnostic ECG Database.
  Entities (basic):        {'Dataset': ['MIT-BIH Arrhythmia Database', 'PTB Diagnostic ECG Database']}
  Entities (w/ desc):      {'Dataset': ['MIT-BIH Arrhythmia Database', 'PTB Diagnostic ECG Database']}

--- Example 4 ---
  Text: Feature extraction was performed using standard NLP tec